In [ ]:
import numpy as np
import netket as nk
import numpy as np
import matplotlib.pyplot as plt
import json
from pyscf import gto, scf, fci
import netket.experimental as nkx
from tqdm import tqdm

bond_distance = np.round(np.linspace(0.4,1.8,15),2)

In [ ]:
H4_E_FCI = []
H4_E_HF = []
H4_nuclear_repulsion = []
for bond_length in tqdm(bond_distance):
    geometry = [
        ('H', (0., 0., 0.)),
        ('H', (bond_length*1, 0., 0.)),
        ('H', (bond_length*2, 0., 0.)),
        ('H', (bond_length*3, 0., 0.)),
    ]

    # 创建分子对象，使用STO-3G基组
    mol = gto.M(atom=geometry, basis='STO-3G')

    # 进行Hartree-Fock计算
    mf = scf.RHF(mol).run(verbose=0)
    E_hf = mf.e_tot
    nuclear_repulsion = mol.energy_nuc()
    H4_nuclear_repulsion.append(nuclear_repulsion)
    
    # 进行FCI计算作为参考
    cisolver = fci.FCI(mf)
    E_fci, fcivec = cisolver.kernel()
    H4_E_FCI.append(E_fci)
    H4_E_HF.append(E_hf)
    
with open('H4_E_FCI_HF.npz', 'wb') as f:
    np.savez(f, E_FCI=H4_E_FCI, E_HF=H4_E_HF,E_nuclear_repulsion=H4_nuclear_repulsion, bond_length=bond_distance)
    

In [ ]:
with open('H4_E_FCI_HF.npz', 'rb') as f:
    data = np.load(f)
    E_FCI = data['E_FCI']      # 不是 E_fci
    E_HF = data['E_HF']
    E_nuclear_repulsion = data['E_nuclear_repulsion']
    bond_length = data['bond_length']

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(bond_length, E_FCI, 'bo-', linewidth=2, markersize=8,label='Full-CI')
plt.plot(bond_length, E_HF, 'ro-', linewidth=2, markersize=8,label='Hartree-Fock')
#plt.plot(bond_length, E_FFNN, 'go-', linewidth=2, markersize=8,label='VMC')
plt.legend()
plt.xlabel('Bond Length (Å)')
plt.ylabel('FCI (Hartree)')

plt.title('Potential Energy Surface of $H_4$')
plt.grid(True, alpha=0.3)
plt.show()

VMC计算

In [ ]:
import itertools

# 排列组合
letters_alpha= [0,1,2,3]
letters_beta=  [4,5,6,7]
combinations_alpha = itertools.combinations(letters_alpha, 2)
combinations_beta =  itertools.combinations(letters_beta, 2)
clusters = list(combinations_alpha) + list(combinations_beta)
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=4,  # 总空间轨道数
    s = 1/2,
    n_fermions_per_spin=(2, 2)  # 每种自旋的电子数
)
g = nk.graph.Graph(edges=clusters)
sa = nk.sampler.MetropolisFermionHop(
    hi, graph=g, n_chains=16, spin_symmetric=True, sweep_size=64
)

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

class FFN_Amplitude(nnx.Module):

    def __init__(self, N: int, alpha: int = 1, *, rngs: nnx.Rngs):
        self.alpha = alpha
        self.linear = nnx.Linear(in_features=N, out_features=alpha * N, rngs=rngs)

    def __call__(self, x: jax.Array):
        y = self.linear(x)
        y = nnx.relu(y)
        return jnp.sum(y, axis=-1)
    

class FFN_Phase(nnx.Module):

    def __init__(self, N: int, alpha: int = 1, *, rngs: nnx.Rngs):
        self.alpha = alpha
        self.linear = nnx.Linear(in_features=N, out_features=alpha * N, rngs=rngs)

    def __call__(self, x: jax.Array):
        y = self.linear(x)
        y = nnx.relu(y)
        return jnp.sum(y, axis=-1)
    
class FFN(nnx.Module):
    def __init__(self,N:int,alpha:int,rngs_amplitude: nnx.Rngs,rngs_phase:nnx.Rngs) -> None:
        self.ffn_amplitude = FFN_Amplitude(N=N,alpha=alpha,rngs=rngs_amplitude)
        self.ffn_phase = FFN_Phase(N=N,alpha=alpha,rngs=rngs_phase)
        
    def __call__(self, x:jax.Array):
        y = self.ffn_amplitude(x) + 1j*self.ffn_phase(x)
        return y        

In [ ]:
ffnn_model = FFN(N=8,alpha=1,rngs_amplitude=nnx.Rngs(0),rngs_phase=nnx.Rngs(1))
vs = nk.vqs.MCState(sa, ffnn_model, n_discard_per_chain=10, n_samples=512)

FFNN的计算结果

In [ ]:
ffnn_energy=[]
for bond_length in tqdm(bond_distance):
    print(f'当前 bond_length: {bond_length:.2f}')
    geometry = [
        ('H', (0., 0., 0.)),
        ('H', (bond_length, 0., 0.)),
        ('H', (bond_length*2, 0., 0.)),
        ('H', (bond_length*3, 0., 0.)),
        
    ]
    # 创建分子对象，使用STO-3G基组
    ffnn_model = FFN(N=8,alpha=1,rngs_amplitude=nnx.Rngs(0),rngs_phase=nnx.Rngs(1))
    vs = nk.vqs.MCState(sa, ffnn_model, n_discard_per_chain=10, n_samples=512)
    mol = gto.M(atom=geometry, basis='STO-3G')
    ha = nkx.operator.from_pyscf_molecule(mol)
    gs = nk.driver.VMC(ha, opt, variational_state=vs, preconditioner=sr)
    gs.run(200)
    ffnn_energy.append(gs.energy.mean)

In [ ]:
ffnn_energy

In [ ]:
with open('H4_FFNN_alpha_1.npz', 'wb') as f:
    np.savez(f, E_FFNN=ffnn_energy, bond_length=bond_distance)

In [ ]:
with open('H4_FFNN_alpha_1.npz', 'rb') as f:
    data = np.load(f)
    E_FFNN = data['E_FFNN']
    bond_length = data['bond_length']
    
with open('H4_E_FCI_HF.npz', 'rb') as f:
    data = np.load(f)
    E_FCI = data['E_FCI']      # 不是 E_fci
    E_HF = data['E_HF']
    E_nuclear_repulsion = data['E_nuclear_repulsion']
    bond_length = data['bond_length']

In [ ]:
E_FFNN

In [ ]:
from cProfile import label
import numpy as np
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(bond_length, E_FCI, 'bo-', linewidth=2, markersize=8,label='Full-CI')
plt.plot(bond_length, E_HF, 'ro-', linewidth=2, markersize=8,label='Hartree-Fock')
plt.plot(bond_length, E_FFNN, 'go-', linewidth=2, markersize=8,label='VMC')
plt.legend()
plt.xlabel('Bond Length (Å)')
plt.ylabel('FCI (Hartree)')

plt.title('Potential Energy Surface of H2')
plt.grid(True, alpha=0.3)
plt.show()